# 停車場預約 AI Agent - Debug 版本

目標：偵測 https://pcc.youparking.com.tw/parkingreserve/#/reservedlist/1  
當 5/23 可以預約時，透過 LINE Messaging API 或 Gmail 通知

**偵錯流程建議：**
1. 先執行 Cell 1~3（設定 + Logger）
2. 執行 Cell 5（`check_and_book`）- 設 `headless=False` 可看到瀏覽器
3. 觀察截圖與 log 輸出，定位問題

In [ ]:
## Cell 1：匯入套件
import json
import logging
import smtplib
from datetime import datetime
from email.mime.text import MIMEText

import requests
from playwright.async_api import async_playwright, TimeoutError as AsyncPlaywrightTimeoutError

print("✅ 套件匯入成功")

In [ ]:
## Cell 2：設定區
# ⚠️  此 notebook 僅供本機偵錯，請勿上傳到 GitHub（含憑證明文）
# 正式執行請使用 parking_agent_v2.py + GitHub Secrets

TARGET_DATE = "05-23"
CHECK_INTERVAL_MINUTES = 5

LINE_CHANNEL_ACCESS_TOKEN = "your_line_token"      # ← 填入後僅在本機使用
LINE_USER_ID              = "your_line_user_id"    # ← 同上

GMAIL_SENDER    = "your_email@gmail.com"           # ← 同上
GMAIL_PASSWORD  = "your_app_password"              # ← 同上
GMAIL_RECIPIENT = "your_email@gmail.com"

DEBUG_HEADLESS = False

print(f"✅ 設定載入，目標日期：'{TARGET_DATE}'")

In [ ]:
## Cell 3：Logger 設定

logging.basicConfig(
    level=logging.DEBUG,   # ← Debug 版改用 DEBUG 等級，看更多細節
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("parking_agent.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,  # 重新執行時重設 handler
)
log = logging.getLogger(__name__)
print("✅ Logger 設定完成（DEBUG 等級）")

In [ ]:
## Cell 4：通知函式

def notify_line(message: str) -> bool:
    if (not LINE_CHANNEL_ACCESS_TOKEN
            or LINE_CHANNEL_ACCESS_TOKEN == "YOUR_LINE_TOKEN"):
        log.warning("LINE Messaging API token 未設定，跳過 LINE 通知")
        return False
    if not LINE_USER_ID or LINE_USER_ID == "YOUR_USER_ID":
        log.warning("LINE User ID 未設定，跳過 LINE 通知")
        return False
    try:
        payload = {
            "to": LINE_USER_ID,
            "messages": [{"type": "text", "text": message}],
        }
        resp = requests.post(
            "https://api.line.me/v2/bot/message/push",
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {LINE_CHANNEL_ACCESS_TOKEN}",
            },
            data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
            timeout=10,
        )
        if resp.status_code == 200:
            log.info("LINE 通知發送成功")
            return True
        else:
            log.error(f"LINE 通知失敗：{resp.status_code} {resp.text}")
            return False
    except Exception as e:
        log.error(f"LINE 通知例外：{e}")
        return False


def notify_gmail(subject: str, body: str) -> bool:
    if not GMAIL_SENDER or GMAIL_SENDER == "your_email@gmail.com":
        log.warning("Gmail 未設定，跳過 Email 通知")
        return False
    try:
        msg = MIMEText(body, "plain", "utf-8")
        msg["Subject"] = subject
        msg["From"]    = GMAIL_SENDER
        msg["To"]      = GMAIL_RECIPIENT
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=10) as server:
            server.login(GMAIL_SENDER, GMAIL_PASSWORD)
            server.sendmail(GMAIL_SENDER, GMAIL_RECIPIENT, msg.as_string())
        log.info("Gmail 通知發送成功")
        return True
    except Exception as e:
        log.error(f"Gmail 通知例外：{e}")
        return False


def send_notifications(message: str):
    notify_line(f"🚗 停車預約通知\n\n{message}")
    notify_gmail(
        subject="🚗 停車場可以預約了！",
        body=(
            f"{message}\n\n"
            f"請前往：https://pcc.youparking.com.tw/parkingreserve/#/reservedlist/1"
        ),
    )

print("✅ 通知函式載入完成")

In [ ]:
## Cell 5：核心檢查函式（async，已驗證 selector）

async def check_and_book() -> bool:
    log.info(f"開始檢查 {TARGET_DATE} 是否可預約...")
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, slow_mo=300)
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1280, "height": 800},
        )
        page = await context.new_page()
        try:
            # 1. 載入首頁
            await page.goto("https://pcc.youparking.com.tw/parkingreserve/#/",
                            wait_until="networkidle", timeout=30_000)
            await page.wait_for_timeout(1500)

            # 2. 點擊「前往」進入預約列表
            await page.get_by_role("link", name="前往").first.click()
            await page.wait_for_timeout(1000)

            # 3. 勾選同意條款
            await page.locator(".v-input--selection-controls__ripple").click()
            await page.wait_for_timeout(500)

            # 4. 點擊「前往預約」
            await page.get_by_role("button", name="前往預約").click()
            await page.wait_for_timeout(2000)
            try:
                await page.wait_for_load_state("networkidle", timeout=10_000)
            except AsyncPlaywrightTimeoutError:
                pass

            # 5. 找到目標日期那一列，判斷狀態
            target_row = page.locator(f"tr:has(td:has-text('{TARGET_DATE}'))").first
            if await target_row.count() == 0:
                log.warning(f"找不到 {TARGET_DATE} 的列，頁面可能尚未開放該日期")
                return False

            full_el  = target_row.locator(":has-text('已滿')").first
            book_el  = target_row.locator("button, a").filter(has_text="預約").first
            is_full     = await full_el.count() > 0
            is_bookable = await book_el.count() > 0

            if is_full:
                log.info(f"❌ {TARGET_DATE} 已滿，下次再試")
                return False
            elif is_bookable:
                log.info(f"✅ {TARGET_DATE} 可以預約！")
                now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                send_notifications(
                    f"【{TARGET_DATE} 停車位可以預約了！】\n"
                    f"偵測時間：{now_str}\n"
                    f"請立即前往：https://pcc.youparking.com.tw/parkingreserve/#/"
                )
                return True
            else:
                log.warning(f"⚠️ {TARGET_DATE} 狀態未知，請手動確認")
                return False

        except AsyncPlaywrightTimeoutError as e:
            log.error(f"頁面操作逾時：{e}")
            return False
        except Exception as e:
            log.error(f"執行時發生例外：{e}", exc_info=True)
            return False
        finally:
            try:
                ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                await page.screenshot(path=f"screenshot_{ts}.png")
            except Exception:
                pass
            await browser.close()

print("✅ check_and_book 載入完成")

In [ ]:
## Cell 6：一鍵執行（執行前確認 Cell 1~5 已執行）

result = await check_and_book()
print(f"\n結果：{'🟢 可預約，通知已發送！' if result else '🔴 尚未開放或已滿'}")

---
## 偵錯執行區
以下 Cell 可單獨執行，逐步確認每個環節是否正常。

In [ ]:
## Debug Cell A：測試 LINE 通知是否正常

result = notify_line("🔔 測試訊息：LINE 通知功能正常！")
print(f"LINE 通知結果：{'成功' if result else '失敗（查看上方 log）'}")

In [ ]:
## Debug Cell B：測試 Gmail 通知是否正常

result = notify_gmail("🔔 測試信件", "停車 Agent Gmail 通知測試正常！")
print(f"Gmail 通知結果：{'成功' if result else '失敗（查看上方 log）'}")

In [ ]:
## Debug Cell C：單次執行測試（同 Cell 6）

result = await check_and_book()
print(f"\n結果：{'🟢 可預約，通知已發送！' if result else '🔴 尚未開放或已滿'}")

---
## 逐步偵錯區：逐 Cell 執行，檢查每個 Selector

> **使用方式：** 依序執行 Step 1 → 2 → 3 → 4 → 5，瀏覽器會保持開啟。  
> 每一步都會印出命中的 selector 與元素狀態，方便確認問題出在哪。  
> 最後執行 Step 5 關閉瀏覽器。

In [ ]:
## Step 1：啟動瀏覽器並載入頁面

from playwright.async_api import async_playwright, TimeoutError as AsyncPlaywrightTimeoutError

_pw = await async_playwright().start()
_browser = await _pw.chromium.launch(headless=False, slow_mo=500)
_context = await _browser.new_context(
    user_agent=(
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    viewport={"width": 1280, "height": 800},
)
_page = await _context.new_page()

await _page.goto(
    "https://pcc.youparking.com.tw/parkingreserve/#/reservedlist/1",
    wait_until="networkidle",
    timeout=30_000,
)
await _page.wait_for_timeout(2000)

print("✅ 頁面載入完成，URL：", _page.url)
print("   頁面標題：", await _page.title())

In [ ]:
## Step 2：點擊「前往」連結，進入預約頁面

# 錄製結果：從首頁點「前往」進入，URL 是 /#/ 而非直接 /#/reservedlist/1
await _page.get_by_role("link", name="前往").first.click()
await _page.wait_for_timeout(1000)

print("✅ 已點擊「前往」，目前 URL：", _page.url)
await _page.screenshot(path="debug_step2_after_goto.png")
print("📸 截圖已存：debug_step2_after_goto.png")

In [ ]:
## Step 2b：勾選同意條款（此步驟可跳過，Step 3 會直接點按鈕）
# 錄製結果：checkbox 是 Vuetify 的 ripple 元素

await _page.locator(".v-input--selection-controls__ripple").click()
await _page.wait_for_timeout(500)
print("✅ 已勾選同意條款")
await _page.screenshot(path="debug_step2b_checkbox.png")
print("📸 截圖已存：debug_step2b_checkbox.png")

In [ ]:
## Step 3：點擊「下一步」，等待頁面更新
# 直接用錄製的 selector，不依賴前一步的變數

await _page.get_by_role("button", name="前往預約").click()
print("✅ 已點擊「前往預約」，等待頁面更新...")
await _page.wait_for_timeout(2000)
try:
    await _page.wait_for_load_state("networkidle", timeout=10_000)
    print("✅ 頁面已穩定（networkidle）")
except AsyncPlaywrightTimeoutError:
    print("⚠️  networkidle 逾時（SPA 正常現象）")

print(f"目前 URL：{_page.url}")
await _page.screenshot(path="debug_step3_after_next.png")
print("📸 截圖已存：debug_step3_after_next.png")

In [ ]:
## Step 4：偵測目標日期是否可預約

# 錄製結果：日期在 table 的 td > span 內
# 位置型 selector（脆弱，日期不同時 nth-child 會變）：tr:nth-child(19) > td:nth-child(2) > span
# 改用文字比對（穩健）：找包含 TARGET_DATE 文字的 td 裡的 span

date_el = _page.locator(f"td:has-text('{TARGET_DATE}') > span, td > span:has-text('{TARGET_DATE}')").first
count = await date_el.count()
print(f"=== 日期元素偵測（目標：{TARGET_DATE}）===")
print(f"  找到元素數量：{count}")

if count > 0:
    visible  = await date_el.is_visible()
    tag      = await date_el.evaluate("el => el.tagName")
    text     = (await date_el.inner_text()).strip()
    classes  = await date_el.evaluate("el => el.className")
    # 取父層 td 的 disabled / class 狀態
    td_el        = _page.locator(f"td:has-text('{TARGET_DATE}')").first
    td_classes   = await td_el.evaluate("el => el.className")
    td_disabled  = await td_el.get_attribute("disabled")
    cls_disabled    = await td_el.evaluate("el => el.classList.contains('disabled')")
    cls_unavailable = await td_el.evaluate("el => el.classList.contains('unavailable')")
    cls_full        = await td_el.evaluate("el => el.classList.contains('full')")

    is_disabled = bool(td_disabled) or cls_disabled or cls_unavailable or cls_full
    status = "🔴 不可預約（disabled）" if is_disabled else "🟢 可預約！"

    print(f"  span tag={tag}, text='{text}', visible={visible}")
    print(f"  span class='{classes}'")
    print(f"  父層 td class='{td_classes}'")
    print(f"  狀態：{status}")
else:
    print(f"  ❌ 找不到含 '{TARGET_DATE}' 的 td > span")
    print("     嘗試廣播搜尋整個頁面...")
    all_matches = await _page.locator(f"*:has-text('{TARGET_DATE}')").all()
    print(f"  廣播找到 {len(all_matches)} 個元素：")
    for el in all_matches[:10]:
        try:
            tag = await el.evaluate("el => el.tagName")
            cls = await el.evaluate("el => el.className")
            txt = (await el.inner_text()).strip()[:40]
            print(f"    {tag}  class='{cls}'  text='{txt}'")
        except Exception:
            pass

await _page.screenshot(path="debug_step4_date.png")
print("\n📸 截圖已存：debug_step4_date.png")

In [ ]:
## Step 4b：找到 5/23 那列，判斷狀態欄是「已滿」還是可預約按鈕
# 錄製結果：
#   日期格式為 "2026-05-23 (六)"，在 role=cell 裡
#   狀態文字「已滿」在同一個 tbody 內

# 1. 找到包含 TARGET_DATE 的那整列 tr
#    頁面實際日期格式：2026-05-23 (六) → 用 has-text 模糊比對月/日即可
target_row = _page.locator(f"tr:has(td:has-text('{TARGET_DATE}'))").first
row_count = await target_row.count()
print(f"找到含 '{TARGET_DATE}' 的 tr：{row_count} 個")

if row_count == 0:
    # 日期格式可能是完整格式，嘗試 role=cell
    target_row = _page.get_by_role("row", name=TARGET_DATE).first
    row_count = await target_row.count()
    print(f"改用 role=row 搜尋：{row_count} 個")

if row_count == 0:
    print("❌ 找不到目標列，印出 tbody 所有文字供參考：")
    tbody_text = await _page.locator("tbody").inner_text()
    print(tbody_text[:800])
else:
    # 2. 印出整列所有 td 內容
    tds = await target_row.locator("td").all()
    print(f"該列共有 {len(tds)} 個 td：")
    for i, td in enumerate(tds):
        try:
            txt = (await td.inner_text()).strip()
            print(f"  td[{i}] = '{txt}'")
        except Exception:
            pass

    # 3. 判斷：同一列內找「已滿」或預約按鈕
    #    錄製顯示「已滿」是純文字 span，非 disabled button
    full_el  = target_row.locator(":has-text('已滿')").first
    book_el  = target_row.locator("button, a").filter(has_text="預約").first

    is_full     = await full_el.count() > 0
    is_bookable = await book_el.count() > 0

    print(f"\n--- 判斷結果 ---")
    if is_full:
        status_text = (await full_el.inner_text()).strip()
        print(f"🔴 狀態：{status_text}（不通知）")
    elif is_bookable:
        btn_text = (await book_el.inner_text()).strip()
        print(f"🟢 狀態：可預約（按鈕='{btn_text}'）→ 發送 LINE / Email 通知！")
        # 取消下方註解可直接測試通知
        # send_notifications(f"【{TARGET_DATE} 停車位可預約！】\n請前往預約：https://pcc.youparking.com.tw/parkingreserve/#/reservedlist/1")
    else:
        print(f"⚠️  未知狀態，請看下方 HTML：")
        row_html = await target_row.evaluate("el => el.outerHTML")
        print(row_html)

await _page.screenshot(path="debug_step4b_row.png")
print("\n📸 截圖已存：debug_step4b_row.png")

In [ ]:
## Step 5：關閉瀏覽器

await _page.screenshot(path="debug_step5_final.png")
print("📸 最終截圖已存：debug_step5_final.png")
await _browser.close()
await _pw.stop()
print("✅ 瀏覽器已關閉")